# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Group work is allowed

You may work on this lab alone or in a group of up to three. Groups talk through the design together, but **every member turns in their own notebook** — one submission per person, not one per group.

**Every submission must name its group.** Fill in the cell below before you do anything else. If you worked alone, say so there. A notebook that does not name its group is treated as unattributed work, and that is a problem we would both rather avoid.

Your notebooks will look similar to each other, and that is fine when the group is named. It is not fine when it is not.

In [ ]:
# Who worked on this notebook?
# List every member, yourself included. Use computing IDs.
# Working alone is fine -- put just your own ID and set SOLO = True.
GROUP_MEMBERS = ['gzw2nc']   # e.g. ['abc1de', 'fgh2ij']
SOLO = len(GROUP_MEMBERS) == 1

assert GROUP_MEMBERS != ['your_id_here'], 'Fill in GROUP_MEMBERS before you submit.'
print('Worked alone.' if SOLO else 'Group: ' + ', '.join(GROUP_MEMBERS))

Worked alone.


### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [ ]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    delay = 1
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            elif r.status_code in {400, 404}:
                r.raise_for_status()
            elif r.status_code in RETRY_STATUSES:
                print(f"Retryable error {r.status_code}. Retrying in {delay}s...")
            else:
                r.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Connection error: {e}. Retrying in {delay}s...")

        time.sleep(delay)
        delay *= 2

    raise RuntimeError("All weather API attempts failed.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [ ]:
# Call get_weather once for a range that spans both dates, then keep just the two dates you care about
used_fallback = False
target_dates = ['2026-09-05', '2026-10-03']

try:
    wx_raw = get_weather(start='2026-09-05', end='2026-10-03')
    wx = wx_raw[wx_raw['time'].isin(target_dates)].copy()
except Exception as e:
    print(f"API failed, falling back to sample data. Error: {e}")
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [ ]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [ ]:
# Rename API columns to clean joinable format, convert datatypes
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [ ]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [ ]:
# Compare precipitation on the two targeted dates
row1 = wx[wx['date'] == '2026-09-05'].iloc[0]
row2 = wx[wx['date'] == '2026-10-03'].iloc[0]

p1, p2 = row1['precip_mm'], row2['precip_mm']

if p1 > p2:
    diff = p1 - p2
    print(f"The game on September 5th, 2026 was wetter than October 3rd, 2026 by {diff:.1f} mm.")
elif p2 > p1:
    diff = p2 - p1
    print(f"The game on October 3rd, 2026 was wetter than September 5th, 2026 by {diff:.1f} mm.")
else:
    print(f"Both game days experienced the same level of precipitation ({p1:.1f} mm).")

The game on October 3rd, 2026 was wetter than September 5th, 2026 by 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [ ]:
# Print ingestion run report & lineage metadata
import datetime
pull_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S EST')

print("=== DATA METADATA CITATION ===")
print(f"Endpoint: {BASE}")
print("Coordinates: latitude=38.03, longitude=-78.51")
print("Date Range: 2026-09-05 to 2026-10-03")
print(f"Pulled at: {pull_time}")
print(f"Data Status: {'Using mock fallback data' if used_fallback else 'Using live open-meteo API data'}")

=== DATA METADATA CITATION ===
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude=38.03, longitude=-78.51
Date Range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-14 18:17:52 EST
Data Status: Using live open-meteo API data


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If this happened, a standard merge/join onto the 200 days of sales data would result in the weather columns becoming entirely empty. If you used an inner join, the merged dataset would have no rows at all, so the chart wouldn't even show anything. If you did a left join, all 200 rows would still be there, but because its filled with NaN for the weather variables, the chart would also be blank or missing data. In both cases, the plotting libraries won't flag the missing data on its own, it will just produce the empty/incomplete plot. Having the exception (raise) is thus important to make sure these things don't happen silently.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [ ]:
# Stretch: Pull Charlottesville & Orlando weather to produce a long-format dataset
try:
    cville_raw = get_weather(start='2026-09-05', end='2026-10-03', lat=38.03, lon=-78.51)
    cville_clean = cville_raw[cville_raw['time'].isin(target_dates)].copy()
    cville_clean['city'] = 'Charlottesville'
    cville_clean['source'] = 'open-meteo'

    orlando_raw = get_weather(start='2026-09-05', end='2026-10-03', lat=28.54, lon=-81.38)
    orlando_clean = orlando_raw[orlando_raw['time'].isin(target_dates)].copy()
    orlando_clean['city'] = 'Orlando'
    orlando_clean['source'] = 'open-meteo'

    stretch_df = pd.concat([cville_clean, orlando_clean], ignore_index=True)
except Exception as e:
    print(f"Failed to query API for stretch target. Loading fallbacks. Error: {e}")
    # Generate simulated combined fallback dataset
    stretch_df = pd.DataFrame({
        'time': ['2026-09-05', '2026-10-03', '2026-09-05', '2026-10-03'],
        'temperature_2m_max': [30.1, 18.4, 32.5, 29.0],
        'precipitation_sum': [0.0, 22.5, 12.0, 0.5],
        'city': ['Charlottesville', 'Charlottesville', 'Orlando', 'Orlando'],
        'source': ['fallback sample', 'fallback sample', 'fallback sample', 'fallback sample']
    })

stretch_df = stretch_df.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
stretch_df['date'] = pd.to_datetime(stretch_df['date'])
stretch_df = stretch_df[['date', 'city', 'temp_max', 'precip_mm', 'source']]
display(stretch_df)

,date,city,temp_max,precip_mm,source
0,2026-09-05,Charlottesville,29.4,2.5,open-meteo
1,2026-10-03,Charlottesville,22.0,14.7,open-meteo
2,2026-09-05,Orlando,32.4,3.9,open-meteo
3,2026-10-03,Orlando,30.4,13.4,open-meteo


### Before you submit

- `GROUP_MEMBERS` is filled in with real computing IDs, and the cell runs without tripping its assert. This is the one thing people forget.
- Name the same people in the submission comment when you upload.
- Run **Kernel → Restart and Run All** and make sure it still works top to bottom.
- The write-up is answered in your own words, not the group's shared draft.